# Task 5: NOVA AI Multi-Agent Platform

**NOVA AI Platform | Assessment Task 5**

LangGraph multi-agent system integrating Tasks 1–4:

```
Customer Query
  → TicketRouter     (Task 1: COSTAR + CoT intent classification)
  → SupportAgent     (Task 2: MCP tools — order status / returns)
  → RAGAgent         (Task 3: hybrid BM25 + ChromaDB search)
  → PersonalizationAgent  (Task 2: recommend_products tool)
  → EscalationAgent  (human-in-the-loop)
  → BrandVoiceAgent  (Task 4: NOVA tone polish)
  → AuditLogger      (nova_traces.json)
```

**3 Demo Scenarios:**
1. Order status — happy path (MCP lookup)
2. Product/ingredient query (RAG pipeline)
3. Angry customer — escalation to human agent

---
⚠️ **Enable GPU**: Runtime → Change runtime type → T4 GPU (needed for sentence-transformers)

## 1. Installation

In [ ]:
# Install all required packages
!pip install langgraph langchain-core openai chromadb sentence-transformers \
    rank-bm25 datasets -q

# Required for loading the fine-tuned TinyLlama brand voice model (Task 4)
!pip uninstall bitsandbytes -y -q
!pip install bitsandbytes --no-cache-dir -q
!pip install transformers==4.44.2 peft==0.12.0 accelerate==0.34.2 -q

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration & API Keys

In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    HF_TOKEN     = userdata.get('HF_TOKEN')
    HF_USERNAME  = userdata.get('HF_USERNAME') or 'simran681'
except Exception:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
    HF_TOKEN     = os.getenv('HF_TOKEN', '')
    HF_USERNAME  = os.getenv('HF_USERNAME', 'simran681')

# Fine-tuned brand voice adapter trained in Task 4 (pushed to HF Hub)
HF_REPO = f'{HF_USERNAME}/nova-brand-voice-tinyllama'
USE_FINETUNED_BRAND_VOICE = True   # Uses Task 4 TinyLlama adapter for brand voice polishing
USE_RAG = True                      # Set False to skip ChromaDB (faster startup)

# Login to HuggingFace so the model can be downloaded
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'HuggingFace login: OK')
else:
    print('WARNING: HF_TOKEN not set — add it to Colab Secrets if the model repo is private')

GITHUB_RAW = 'https://raw.githubusercontent.com/simran681/nova-ai-platform/main'

print(f'GROQ_API_KEY set            : {bool(GROQ_API_KEY)}')
print(f'USE_RAG                     : {USE_RAG}')
print(f'USE_FINETUNED_BRAND_VOICE   : {USE_FINETUNED_BRAND_VOICE}')
print(f'Fine-tuned model repo       : {HF_REPO}')

## 3. Download Project Files from GitHub

In [ ]:
import urllib.request, os

def fetch(filename, url):
    os.makedirs(os.path.dirname(filename), exist_ok=True) if os.path.dirname(filename) else None
    if not os.path.exists(filename):
        print(f'Downloading {filename}...')
        urllib.request.urlretrieve(url, filename)
        print(f'  Saved {filename}')
    else:
        print(f'  {filename} already exists')

fetch('nova_mock_db.json',          f'{GITHUB_RAW}/nova_mock_db.json')
fetch('rag_module.py',              f'{GITHUB_RAW}/rag_module.py')
fetch('task5_nova_platform.py',     f'{GITHUB_RAW}/task5_nova_platform.py')
fetch('task2_mcp/__init__.py',      f'{GITHUB_RAW}/task2_mcp/__init__.py')
fetch('task2_mcp/client.py',        f'{GITHUB_RAW}/task2_mcp/client.py')
print('\nAll files ready')

## 4. Initialize NOVA Platform

In [ ]:
import sys
sys.path.insert(0, '.')

from task5_nova_platform import NOVAPlatform, NOVAPlatformConfig

config = NOVAPlatformConfig(
    groq_api_key=GROQ_API_KEY,
    llm_model='llama-3.1-8b-instant',
    mock_db_path='nova_mock_db.json',
    chroma_path='./chroma_db',
    audit_log_path='nova_traces.json',
    finetuned_model_path=HF_REPO if USE_FINETUNED_BRAND_VOICE else None,
    use_rag=USE_RAG,
    use_brand_voice=True
)

platform = NOVAPlatform(config)
print('\nPlatform ready!')

## 5. Visualize Agent Graph

In [ ]:
from IPython.display import Image, display

try:
    img = platform.visualize_graph('nova_agent_graph.png')
    display(img)
except Exception as e:
    print(f'PNG generation failed: {e}')
    print('\nAgent Graph (Mermaid):')
    print('''
graph TD
    START --> TicketRouter
    TicketRouter -- order_status/return --> SupportAgent
    TicketRouter -- product_query/sizing --> RAGAgent
    TicketRouter -- recommendation --> PersonalizationAgent
    TicketRouter -- escalate/injection --> EscalationAgent
    SupportAgent --> BrandVoiceAgent
    RAGAgent --> BrandVoiceAgent
    PersonalizationAgent --> BrandVoiceAgent
    EscalationAgent --> AuditLogger
    BrandVoiceAgent --> AuditLogger
    AuditLogger --> END
''')

## 6. Scenario 1 — Order Status (Happy Path)

**Expected flow:** `TicketRouter → SupportAgent (get_order_status via MCP) → BrandVoiceAgent → AuditLogger`

In [ ]:
result1 = platform.process_ticket(
    ticket="Hi! My order ORD-1042 was placed a week ago but I haven't received any updates. "
           "Can you tell me where it is? My tracking number isn't working.",
    customer_id='CUST-1010',
    session_id='demo-scenario-1'
)

print(f'\n📊 Scenario 1 Results:')
print(f'  Intent    : {result1["intent"]}')
print(f'  Escalated : {result1["escalated"]}')
print(f'  MCP calls : {result1["tool_calls"]}')
print(f'  Audit     : {result1["audit_trail_length"]} entries')

## 7. Scenario 2 — Product / Ingredient Query (RAG)

**Expected flow:** `TicketRouter → RAGAgent (BM25 + ChromaDB + reranker) → BrandVoiceAgent → AuditLogger`

In [ ]:
result2 = platform.process_ticket(
    ticket="I'm pregnant and want to know which NOVA skincare products are safe to use. "
           "Specifically, does the Glow Serum contain retinol? And what about the Vitamin C serum — "
           "is it safe during pregnancy?",
    customer_id='CUST-1005',
    session_id='demo-scenario-2'
)

print(f'\n📊 Scenario 2 Results:')
print(f'  Intent    : {result2["intent"]}')
print(f'  Escalated : {result2["escalated"]}')
print(f'  Audit     : {result2["audit_trail_length"]} entries')

## 8. Scenario 3 — Angry Customer Escalation

**Expected flow:** `TicketRouter (detect high frustration + legal threat) → EscalationAgent → AuditLogger`

In [ ]:
result3 = platform.process_ticket(
    ticket="THIS IS ABSOLUTELY OUTRAGEOUS!!! I've contacted your useless support team "
           "FOUR TIMES about my missing order and nobody has helped me. My solicitor "
           "is already aware of this situation and I will be going to Trading Standards "
           "if this is not resolved TODAY. I'm also posting about this on every social "
           "media platform I can find. Sort it OUT!!!",
    customer_id='CUST-1020',
    session_id='demo-scenario-3'
)

print(f'\n📊 Scenario 3 Results:')
print(f'  Intent    : {result3["intent"]}')
print(f'  Escalated : {"✅ YES — Human agent notified" if result3["escalated"] else "❌ NOT escalated (unexpected)"}')
print(f'  Audit     : {result3["audit_trail_length"]} entries')
print()
print('  In production this would:')
print('    1. Alert senior support agent dashboard')
print('    2. Attach full context summary + order history')
print('    3. Trigger 15-minute SLA response timer')

## 9. Bonus — Recommendation Scenario

In [ ]:
result4 = platform.process_ticket(
    ticket='I have oily skin and I am looking for a new moisturizer and serum. '
           'What would you recommend from NOVA?',
    customer_id='CUST-1003',
    session_id='demo-scenario-4'
)

print(f'\n📊 Recommendation Scenario Results:')
print(f'  Intent    : {result4["intent"]}')
print(f'  Escalated : {result4["escalated"]}')
print(f'  Audit     : {result4["audit_trail_length"]} entries')

## 10. Audit Trail Summary — nova_traces.json

In [ ]:
import json

print('=' * 65)
print('  NOVA AI PLATFORM — SESSION AUDIT SUMMARY')
print('=' * 65)

try:
    with open('nova_traces.json') as f:
        traces = [json.loads(l) for l in f if l.strip()]

    print(f'\nTotal sessions logged: {len(traces)}')

    for trace in traces:
        print(f'\n  Session   : {trace["session_id"]}')
        print(f'  Ticket    : {trace["ticket"][:70]}...')
        print(f'  Intent    : {trace.get("intent")} ({trace.get("intent_confidence", 0):.0%})')
        print(f'  Escalated : {trace.get("escalated")}')
        print(f'  Tools     : {len(trace.get("tool_calls", []))} MCP calls')
        print(f'  Context   : {trace.get("context_retrieved", 0)} RAG chunks')
        print(f'  Audit     : {len(trace.get("audit_trail", []))} trail entries')
        print(f'  Timestamp : {trace.get("timestamp")}')
        print(f'  Response  : {trace.get("final_response", "")[:100]}...')

except FileNotFoundError:
    print('nova_traces.json not found — run scenarios first')

## 11. Download nova_traces.json

In [ ]:
from google.colab import files
import os

if os.path.exists('nova_traces.json'):
    files.download('nova_traces.json')
    print('nova_traces.json downloaded')
else:
    print('nova_traces.json not found — run the scenario cells first')